# 03 · Check the Full image inventory

**Question:** Does every training image named in the metadata have both expected image files, with no missing, extra or duplicated names?

**Inputs:** `data/raw/train.csv`, `data/raw/jpeg/train` and `data/raw/train`. **Output:** an inventory table and a pass/fail check; no files are changed. JPEG is a common image format; DICOM is a medical-imaging format. Matching names does not prove their contents agree.

This supports [Full cleaning](01_data_cleaning.ipynb) and [Full exploration](02_data_exploration.ipynb). Check filenames before image modelling so missing files are found early; opening and visually checking the images is a separate step.

## Setup

Set `RAW_DATA_DIR` to the folder containing the full dataset. JPEG and DICOM are two file formats for the same source images. This check expects one file of each format for every image name in `train.csv`.

In [1]:
from pathlib import Path

from melanoma_detection.image_validation import check_image_inventory


# Change this one path to your local raw-data directory.
RAW_DATA_DIR = Path(
    r"C:/Users/thoma/Documents/GitHub/ADS2002-Melanoma/data/raw"
)

TRAIN_CSV = RAW_DATA_DIR / "train.csv"
JPEG_TRAIN_DIR = RAW_DATA_DIR / "jpeg" / "train"
DICOM_TRAIN_DIR = RAW_DATA_DIR / "train"

print(f"CSV:   {TRAIN_CSV}")
print(f"JPEG:  {JPEG_TRAIN_DIR}")
print(f"DICOM: {DICOM_TRAIN_DIR}")


CSV:   C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\raw\train.csv
JPEG:  C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\raw\jpeg\train
DICOM: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\raw\train


## Compare the metadata with both image folders

The check matches the image name in the CSV with each filename, ignoring the extension. It also looks for files that have no matching CSV row and names that appear more than once.

In [2]:
report = check_image_inventory(
    train_csv=TRAIN_CSV,
    jpeg_directory=JPEG_TRAIN_DIR,
    dicom_directory=DICOM_TRAIN_DIR,
)


## Results

### Read the count table

The first four rows show how many metadata rows, unique image names and files were found. Matching totals are useful, but totals alone could hide a missing image paired with an extra image. The remaining rows check those mismatches directly; all of those counts should be zero.

There is no graph here because the question is an exact inventory check. The table tells us which check failed and how many filenames need attention.

In [3]:
print(f"{'Check':<32} {'Count':>8}")
print("-" * 41)
for check_name, count in report.summary_rows():
    print(f"{check_name:<32} {count:>8,}")


Check                               Count
-----------------------------------------
CSV rows                           33,126
unique CSV image names             33,126
JPEG files                         33,126
DICOM files                        33,126
duplicated CSV image names              0
CSV names missing JPEG                  0
CSV names missing DICOM                 0
JPEG files absent from CSV              0
DICOM files absent from CSV             0
duplicate JPEG stems                    0
duplicate DICOM stems                   0


### Inspect any mismatched names

If a mismatch count is above zero, the output below lists up to 20 example names. A missing name means the CSV expects a file that was not found. A file absent from the CSV may be an extra file or belong to another dataset. These examples help locate the issue; a list of 20 is a sample, not the full count.

In [4]:
for category, examples in report.mismatch_examples(limit=20).items():
    print(f"{category} ({len(examples)} shown): {list(examples)}")


duplicate_csv_names (0 shown): []
missing_jpeg (0 shown): []
missing_dicom (0 shown): []
orphan_jpeg (0 shown): []
orphan_dicom (0 shown): []
duplicate_jpeg_stems (0 shown): []
duplicate_dicom_stems (0 shown): []


## Check that the inventory passes

Both checks below should be `True`. The strict check also rejects duplicate names and extra files. Passing confirms that the filenames match; it does not confirm that the images can be opened or that their labels are correct.

In [5]:
print(
    "Every CSV image has JPEG and DICOM:",
    report.all_csv_rows_have_both_formats,
)
print("Directories exactly match train.csv:", report.directories_exactly_match_csv)

# strict=True also rejects duplicate identifiers/stems and orphan image files.
report.assert_complete(strict=True)
print("All image-inventory checks passed.")


Every CSV image has JPEG and DICOM: True
Directories exactly match train.csv: True
All image-inventory checks passed.


## Reuse the check

The same function can check image folders from another notebook once their paths are set.

```python
from melanoma_detection.image_validation import check_image_inventory

report = check_image_inventory(TRAIN_CSV, JPEG_TRAIN_DIR, DICOM_TRAIN_DIR)
report.assert_complete(strict=True)
```

## Limitations

This check compares filenames. It does not open each image, detect corrupted contents, compare the appearance of duplicate images, or verify the diagnosis attached to an image. It also does not check whether the JPEG and DICOM versions show the same lesion.

The saved result describes the folders present when the notebook ran. Changing those folders or the CSV requires another check. No model is trained, so this notebook cannot establish prediction performance.

## Summary

The saved check found 33,126 training images, 33,126 JPEG files and 33,126 DICOM files. It found no missing files, extra files or duplicate names, and the strict inventory check passed.

Keep this check as a preparation step. There is no model to retain at this stage. Before image training, check that the files can be opened as well as matched by name, then continue with the appropriate dataset's image workflow.

Return to the [notebook index](../README.md) to choose a workflow. Full-dataset model training is not implemented here.